# ALMA target explorer (Julia)

Pick one of **100 protoplanetary-disk targets** (listed alphabetically, each tagged by how TW Hya-like it is: T Tauri analogue, cavity, young/embedded, Herbig, non-axisymmetric, edge-on, outbursting), or any custom source, query the ALMA archive, download a continuum product, and run the deprojected radial-profile and ring/gap analysis — generalised from the TW Hya notebook.

**Workflow:** menu → `select_target` → ALMA TAP query → observation table (finest resolution first) → size-guarded download → FITS image → centre fit → radial profile → Savitzky–Golay baseline subtraction → rings/gaps → literature comparison (where the catalogue has one).

**Quick start:** run all cells down to *Select and run*. Leave `TARGET_CHOICE = nothing` to be prompted with the menu, or set a number/name (`"MWC 480"`, or a menu number; numbers are alphabetical) or `custom_target("name"; distance_pc = ...)`.

**Caveat:** only the first-listed 25 well-studied disks (TW Hya, AS 209, HD 163296, ...) have stored coordinates, inclination and PA; the other 75 are resolved by name (CDS Sesame) and have approximate distances only. The catalogue's distances, inclinations and position angles are approximate literature values — verify them before quoting results. Unknown inclination/PA (`?`) means the disk is treated as face-on. See the changelog in the setup cell for what changed from the TW Hya notebook (notably a deprojection fix).

In [ ]:
# alma_target_explorer_julia.ipynb
#
# Generic ALMA continuum explorer: pick one of 100 protoplanetary-disk targets
# (or any custom target), query the ALMA archive, download a continuum
# product, and run the radial-profile / ring-gap analysis.
#
# Changelog
#   v1  Generalised from alma_twhya_query_julia (v1):
#       - 25-target menu + custom targets (CDS Sesame name resolution)
#       - observation chooser (finest spatial resolution first, optional band)
#       - tarball size guard before downloading
#       - FIXED deprojection geometry (the TW Hya v1 code stretched the major
#         axis instead of the minor axis; negligible at i = 7 deg, wrong for
#         inclined disks). PA is now east of north on the sky.
#       - star centre refined by minimising within-annulus scatter instead of
#         using the brightest pixel (which is wrong for ring/cavity disks)
#       - beam-based radial bin width, BUNIT-aware labels
#   v2  - menu expanded from 25 to 100 disks, listed alphabetically (print_target_menu(by = :group)
#         restores the TW Hya-likeness grouping); new groups: edge-on, outbursting
#       - the 75 added targets have no stored coordinates: CDS Sesame resolves them when selected
#       - the TW Hya default for the prompt is kept (Enter = TW Hya)
#       - archive top list: 100 targets, name variants merged, OFF/Sun dropped, curated entries kept
#
# Packages go into a local project environment (./alma_env); the first run
# downloads and precompiles everything (CairoMakie takes a few minutes).

import Pkg
Pkg.activate(joinpath(pwd(), "alma_env"))

let needed = ["HTTP", "EzXML", "CSV", "DataFrames", "FITSIO", "CairoMakie"]
    installed = keys(Pkg.project().dependencies)
    to_add = filter(p -> !(p in installed), needed)
    if !isempty(to_add)
        println("Installing: ", join(to_add, ", "))
        Pkg.add(to_add)
    end
end

using HTTP, EzXML, CSV, DataFrames, FITSIO, CairoMakie
using Downloads, Tar                      # stdlib
using Statistics, LinearAlgebra, Printf   # stdlib

println("Julia ", VERSION, " | environment: ", dirname(Base.active_project()))

## Configuration and plot theme

In [ ]:
const CACHE_DIR  = "./alma_cache"
const OUTPUT_DIR = "./alma_plots"
mkpath(CACHE_DIR)
mkpath(OUTPUT_DIR)

# Step-by-step checkpoints inside the radial-profile functions.
const DEBUG = Ref(true)
# Also write every figure to OUTPUT_DIR as a PNG.
const SAVE_PLOTS = Ref(true)
# Refuse to download a product tarball larger than this (GB) unless forced.
const MAX_TAR_GB = Ref(15.0)

_dbg(args...) = DEBUG[] && println("[DEBUG] ", join(string.(args), " "))

# ALMA's TAP/datalink services intermittently return transient 502/503
# errors; the regional mirrors serve the same archive, so we retry with
# backoff and rotate mirrors.
const ALMA_MIRRORS = [
    "https://almascience.nrao.edu",
    "https://almascience.eso.org",
    "https://almascience.nao.ac.jp",
]

# Dark, publication-style theme (#111111 background, #EEEEEE text).
const BG_COLOR = "#111111"
const FG_COLOR = "#EEEEEE"
const C_FLUX   = "#FF9F43"   # azimuthal mean
const C_RING   = "#FFD43B"   # rings
const C_GAP    = "#5DADE2"   # gaps
const C_RESID  = "#4EA1FF"   # residual curve
const C_LIT    = "#FF4D6D"   # literature gap markers

function alma_dark_theme!()
    set_theme!(theme_dark())
    update_theme!(
        backgroundcolor = BG_COLOR,
        Axis = (
            backgroundcolor = BG_COLOR,
            xgridcolor = (FG_COLOR, 0.10), ygridcolor = (FG_COLOR, 0.10),
            xlabelcolor = FG_COLOR, ylabelcolor = FG_COLOR, titlecolor = FG_COLOR,
            xticklabelcolor = FG_COLOR, yticklabelcolor = FG_COLOR,
            xtickcolor = FG_COLOR, ytickcolor = FG_COLOR,
            leftspinecolor = FG_COLOR, rightspinecolor = FG_COLOR,
            topspinecolor = FG_COLOR, bottomspinecolor = FG_COLOR,
        ),
        Legend = (backgroundcolor = BG_COLOR, framecolor = (FG_COLOR, 0.4), labelcolor = FG_COLOR),
        Colorbar = (labelcolor = FG_COLOR, ticklabelcolor = FG_COLOR, tickcolor = FG_COLOR),
    )
end
alma_dark_theme!()

function _save_fig(fig, name)
    SAVE_PLOTS[] || return nothing
    path = joinpath(OUTPUT_DIR, name)
    save(path, fig; px_per_unit = 2)
    println("Saved figure: ", path)
    return path
end

## Target catalogue and menu

In [ ]:
# ---------------------------------------------------------------------------
# Target catalogue: 100 widely studied ALMA disks, listed alphabetically
# ---------------------------------------------------------------------------
# IMPORTANT: distances, inclinations, position angles, spectral-type groupings
# and literature gap radii are approximate values entered from memory (DSHARP,
# MAPS, ALMA Partnership 2015, Gaia). Distances/inclinations/PAs only drive the
# au scaling and deprojection; VERIFY them before quoting results and override
# per run with `incl_deg=`, `pa_deg=`, `distance_pc=`. `?` in the menu =
# unknown, in which case the disk is treated as face-on. Literature gap radii
# (`lit`) exist only for TW Hya, HL Tau and HD 163296.
#
# Groups (how well the TW Hya-style azimuthally averaged analysis applies):
#   :analogue  T Tauri stars (K-M), mostly axisymmetric rings and gaps
#   :cavity    T Tauri stars with large inner cavities (transition disks)
#   :young     younger / embedded (envelope remnants)
#   :herbig    Herbig (A/B/F) stars: more luminous, more massive
#   :asym      strongly non-axisymmetric dust (horseshoes, arcs, shadows)
#   :edge      edge-on disks (cannot be deprojected; radial profile is not a ring/gap plot)
#   :outburst  outbursting (FU Ori / EXor) disks
#
# The first 25 entries (TW Hya ... HD 143006) carry coordinates, inclination and PA.
# The 75 added entries are lighter: approximate distance only; their coordinates
# are resolved with CDS Sesame when you select them (needs network), and inclination/PA
# are unknown (treated face-on) -- pass incl_deg= / pa_deg= from the literature.

_norm(s::AbstractString) = replace(lowercase(s), r"[^a-z0-9]" => "")

const GROUP_LABEL = Dict(:analogue => "T Tauri analogue", :cavity => "T Tauri + cavity",
                         :young => "Young/embedded", :herbig => "Herbig",
                         :asym => "Non-axisymmetric", :edge => "Edge-on",
                         :outburst => "Outbursting", :custom => "Custom")
const GROUP_DIFFERS = Dict(
    :analogue => "Closest analogue: axisymmetric rings/gaps",
    :cavity   => "Ring, not centrally peaked; gaps = cavity physics",
    :young    => "Class I/II, remnant envelope; younger; inclined",
    :herbig   => "Higher-mass star; warmer, larger disk",
    :asym     => "Azimuthal average misleading (horseshoe/arc)",
    :edge     => "Edge-on: cannot deproject; no ring/gap profile",
    :outburst => "Outbursting FU Ori/EXor: hot, bright, young",
    :custom   => "",
)
const GROUP_ORDER = [:analogue, :cavity, :young, :herbig, :asym, :edge, :outburst]

Base.@kwdef struct AlmaTarget
    name::String
    slug::String = replace(name, r"[^A-Za-z0-9]+" => "_")
    keys::Vector{String} = String[]                # normalised name aliases for target_name matching
    ra_deg::Float64 = NaN                          # NaN -> resolved by Sesame on selection
    dec_deg::Float64 = NaN
    distance_pc::Float64 = 100.0
    distance_known::Bool = true
    incl_deg::Union{Nothing,Float64} = nothing
    pa_deg::Union{Nothing,Float64} = nothing       # east of north, major axis
    rmax_arcsec::Float64 = 2.0                     # outer radius of the radial profile
    proposal_id::Union{Nothing,String} = nothing
    lit::Vector{Tuple{Float64,Symbol}} = Tuple{Float64,Symbol}[]   # (radius_au, kind)
    lit_distance_pc::Union{Nothing,Float64} = nothing
    note::String = ""
    group::Symbol = :custom
    differs::String = ""                           # how it differs from TW Hya (menu column)
    simbad::String = ""                           # name(s) for Sesame, '|'-separated (default: name)
end

_T(name, ra, dec, d; aliases = String[], group = :custom, differs = nothing, kw...) =
    AlmaTarget(; name = name, keys = unique([_norm(name); _norm.(aliases)]),
               ra_deg = ra, dec_deg = dec, distance_pc = d, group = group,
               differs = differs === nothing ? GROUP_DIFFERS[group] : differs, kw...)

# coordinate-less entry: Sesame resolves it when selected
_N(name, d; kw...) = _T(name, NaN, NaN, d; kw...)

# natural alphabetical sort key: "HD 97048" before "HD 100546"
_sortkey(s::AbstractString) = replace(lowercase(s), r"\d+" => m -> lpad(m, 6, '0'))

const ALMA_TARGETS = [
    # ---- T Tauri analogues ------------------------------------------------
    _T("TW Hya", 165.466273, -34.704731, 60.1; group = :analogue,
       differs = "Reference: nearest, face-on, ~8-10 Myr, compact",
       incl_deg = 7.0, pa_deg = 155.0, rmax_arcsec = 2.5,
       proposal_id = "2015.1.00686.S", lit_distance_pc = 54.0,
       lit = Tuple{Float64,Symbol}[(1.0, :hole), (22.0, :deep), (37.0, :deep), (6.0, :shallow), (28.0, :shallow), (44.0, :shallow)],
       note = "Closest disk; multi-ring (Andrews+16, 2015.1.00686.S)"),
    _T("AS 209", 252.313700, -14.369100, 121.0; group = :analogue,
       incl_deg = 35.0, pa_deg = 85.8, rmax_arcsec = 2.0,
       note = "Multi-ring disk (DSHARP 2016.1.00484.L, MAPS 2018.1.01055.L)"),
    _T("IM Lup", 239.038300, -37.935000, 158.0; group = :analogue,
       incl_deg = 47.5, pa_deg = 144.5, rmax_arcsec = 2.5,
       note = "Large disk with spirals (DSHARP, MAPS)"),
    _T("GW Lup", 236.686300, -34.510000, 155.0; group = :analogue,
       incl_deg = 38.7, pa_deg = 37.7, rmax_arcsec = 1.5,
       note = "Compact ring + gap (DSHARP)"),
    _T("RU Lup", 239.176300, -37.821000, 159.0; group = :analogue,
       incl_deg = 18.8, pa_deg = 121.0, rmax_arcsec = 1.5,
       note = "Multi-ring disk (DSHARP)"),
    _T("DM Tau", 68.203000, 18.169400, 145.0; group = :analogue,
       incl_deg = 35.0, pa_deg = 157.0, rmax_arcsec = 2.0,
       note = "Disk with inner cavity (approximate parameters)"),
    _T("V4046 Sgr", 273.543700, -32.792900, 72.0; group = :analogue,
       differs = "Binary star; circumbinary disk",
       incl_deg = 34.0, pa_deg = 76.0, rmax_arcsec = 1.5,
       note = "Circumbinary disk"),
    _T("Elias 24", 246.600300, -24.270400, 136.0; group = :analogue, rmax_arcsec = 1.8,
       note = "Wide gap (DSHARP 2016.1.00484.L)"),
    _T("Elias 27", 246.687600, -24.385500, 116.0; group = :analogue,
       differs = "Spirals; less suited to an azimuthal average",
       incl_deg = 56.2, pa_deg = 118.8, rmax_arcsec = 3.0,
       note = "Grand-design spirals (DSHARP)"),
    _T("DoAr 25", 246.571000, -24.720500, 138.0; group = :analogue,
       differs = "Highly inclined; less suited to deprojection",
       rmax_arcsec = 1.5, note = "Multi-ring, highly inclined (DSHARP)"),
    # ---- T Tauri with large cavities --------------------------------------
    _T("GM Aur", 73.795800, 30.366500, 159.0; group = :cavity,
       incl_deg = 53.2, pa_deg = 57.2, rmax_arcsec = 2.0,
       note = "Large cavity (MAPS)"),
    _T("LkCa 15", 69.824200, 22.351000, 157.0; group = :cavity,
       incl_deg = 50.0, pa_deg = 60.0, rmax_arcsec = 1.8,
       note = "Wide cavity"),
    _T("PDS 70", 212.042300, -41.397900, 112.0; group = :cavity,
       incl_deg = 49.7, pa_deg = 158.6, rmax_arcsec = 1.5,
       note = "Cavity with two embedded protoplanets"),
    # ---- Younger / embedded -----------------------------------------------
    _T("HL Tau", 67.910167, 18.232694, 140.0; group = :young,
       incl_deg = 46.7, pa_deg = 138.0, rmax_arcsec = 1.6, lit_distance_pc = 140.0,
       lit = Tuple{Float64,Symbol}[(13.0, :gap), (32.0, :gap), (42.0, :gap), (50.0, :gap), (63.0, :gap), (74.0, :gap)],
       note = "Iconic multi-gap disk (2014 Long Baseline SV; 140 pc adopted)"),
    # ---- Herbig stars -----------------------------------------------------
    _T("HD 163296", 269.088700, -21.956100, 101.0; group = :herbig,
       incl_deg = 46.7, pa_deg = 133.3, rmax_arcsec = 3.0, lit_distance_pc = 101.0,
       lit = Tuple{Float64,Symbol}[(10.0, :gap), (48.0, :gap), (86.0, :gap)],
       note = "Rings and gaps (DSHARP 2016.1.00484.L, MAPS 2018.1.01055.L)"),
    _T("MWC 480", 74.692800, 29.843600, 162.0; group = :herbig,
       incl_deg = 37.0, pa_deg = 148.0, rmax_arcsec = 2.0, note = "Herbig Ae disk (MAPS)"),
    _T("HD 169142", 276.124100, -29.780400, 114.0; group = :herbig,
       differs = "Herbig, but near face-on with rings: closest of its group",
       incl_deg = 13.0, pa_deg = 5.0, rmax_arcsec = 1.5, note = "Near face-on ring + gap system"),
    _T("HD 97048", 167.013800, -77.654900, 185.0; group = :herbig,
       incl_deg = 41.0, pa_deg = 3.0, rmax_arcsec = 3.0, aliases = ["CU Cha"],
       note = "Large multi-ring Herbig disk"),
    _T("HD 100546", 173.106000, -70.194800, 108.0; group = :herbig, rmax_arcsec = 2.0,
       note = "Embedded planet candidates"),
    _T("HD 135344B", 228.951900, -37.154400, 135.0; group = :herbig, rmax_arcsec = 1.5,
       aliases = ["SAO 206462"], note = "Spiral arms + cavity"),
    _T("MWC 758", 82.614700, 25.332500, 160.0; group = :herbig, rmax_arcsec = 2.0,
       note = "Cavity + spirals"),
    _T("AB Aur", 73.941000, 30.551200, 156.0; group = :herbig, rmax_arcsec = 3.0,
       note = "Spirals, protoplanet candidate"),
    # ---- Strongly non-axisymmetric ----------------------------------------
    _T("Oph IRS 48", 246.904900, -24.509700, 134.0; group = :asym, rmax_arcsec = 1.2,
       aliases = ["IRS 48", "WLY 2-48"], note = "Strongly asymmetric dust trap"),
    _T("HD 142527", 239.174500, -42.323100, 157.0; group = :asym, rmax_arcsec = 3.0,
       note = "Horseshoe dust trap (not axisymmetric)"),
    _T("HD 143006", 239.653800, -22.954200, 165.0; group = :asym,
       differs = "Partly asymmetric (rings + shadows)", rmax_arcsec = 1.5,
       note = "Rings, asymmetries, shadows (DSHARP)"),
    # ---- 75 more disks (coordinates resolved by CDS Sesame at selection;
    #      distances approximate; inclination/PA unknown -> treated face-on) ----
    _N("AS 205", 128.0; group = :analogue, differs = "Close binary with disks; otherwise analogue"),
    _N("DoAr 33", 140.0; group = :analogue),
    _N("Elias 20", 138.0; group = :analogue, simbad = "Elias 2-20|[EC92] 20", aliases = ["Elias 2-20"]),
    _N("SR 4", 134.0; group = :analogue, simbad = "SR 4|EM* SR 4"),
    _N("Sz 65", 155.0; group = :analogue),
    _N("Sz 98", 156.0; group = :analogue, differs = "Large, warped/inclined; otherwise analogue", simbad = "Sz 98|HK Lup", aliases = ["HK Lup"]),
    _N("Sz 100", 137.0; group = :analogue),
    _N("Sz 111", 158.0; group = :analogue),
    _N("Sz 114", 162.0; group = :analogue),
    _N("Sz 129", 160.0; group = :analogue),
    _N("V1094 Sco", 153.0; group = :analogue, differs = "Very large disk with wide rings"),
    _N("WaOph 6", 123.0; group = :analogue, simbad = "WaOph 6|V* V2062 Oph"),
    _N("WSB 52", 135.0; group = :analogue),
    _N("AA Tau", 136.0; group = :analogue, differs = "Moderately inclined; warped inner disk"),
    _N("BP Tau", 128.0; group = :analogue),
    _N("CI Tau", 158.0; group = :analogue),
    _N("CY Tau", 128.0; group = :analogue),
    _N("DG Tau", 121.0; group = :analogue, differs = "Jet-driving, younger Class II; envelope traces"),
    _N("DH Tau", 135.0; group = :analogue, differs = "Wide companion; small disk"),
    _N("DL Tau", 159.0; group = :analogue),
    _N("DN Tau", 128.0; group = :analogue),
    _N("DR Tau", 195.0; group = :analogue, differs = "Younger; strong accretion; more distant"),
    _N("DS Tau", 159.0; group = :analogue),
    _N("FP Tau", 128.0; group = :analogue, differs = "Low-mass M star; faint disk"),
    _N("FT Tau", 127.0; group = :analogue),
    _N("GO Tau", 144.0; group = :analogue),
    _N("HP Tau", 161.0; group = :analogue),
    _N("IP Tau", 130.0; group = :analogue),
    _N("IQ Tau", 131.0; group = :analogue),
    _N("SU Aur", 158.0; group = :analogue, differs = "G-type, ~2 Msun; envelope/tail structure"),
    _N("T Tau", 144.0; group = :analogue, differs = "Triple system; disk not isolated"),
    _N("UZ Tau", 131.0; group = :analogue, differs = "Multiple star; circumbinary disk"),
    _N("XZ Tau", 140.0; group = :analogue, differs = "Binary with interacting disks"),
    _N("TWA 7", 34.0; group = :analogue, differs = "Nearer than TW Hya but faint M dwarf disk"),
    _N("PDS 66", 98.0; group = :analogue, differs = "Older (~10-20 Myr), near face-on", simbad = "PDS 66|MP Mus", aliases = ["MP Mus"]),
    _N("CS Cha", 160.0; group = :cavity),
    _N("GG Tau A", 150.0; group = :cavity, differs = "Binary; circumbinary ring", simbad = "GG Tau A|GG Tau", aliases = ["GG Tau"]),
    _N("RX J1604.3-2130A", 150.0; group = :cavity, differs = "Upper Sco; ~10 Myr, big cavity", simbad = "RX J1604.3-2130A|RX J1604.3-2130", aliases = ["RXJ1604"]),
    _N("RX J1615.3-3255", 156.0; group = :cavity, simbad = "RX J1615.3-3255|RX J1615.3-3255", aliases = ["RXJ1615"]),
    _N("RX J1852.3-3700", 146.0; group = :cavity, aliases = ["RXJ1852"]),
    _N("SR 21", 138.0; group = :cavity, simbad = "SR 21|EM* SR 21"),
    _N("SR 24S", 114.0; group = :cavity, differs = "Ring, not centrally peaked; companion SR 24N", simbad = "SR 24S|EM* SR 24", aliases = ["SR 24"]),
    _N("Sz 91", 159.0; group = :cavity),
    _N("T Cha", 107.0; group = :cavity, differs = "Transition disk; ~7 Myr; small ring"),
    _N("UX Tau A", 140.0; group = :cavity, differs = "Multiple star; cavity ~ 25 au", simbad = "UX Tau A|UX Tau", aliases = ["UX Tau"]),
    _N("WSB 60", 135.0; group = :cavity),
    _N("CQ Tau", 162.0; group = :herbig, differs = "Herbig F star; ring and cavity"),
    _N("HD 100453", 104.0; group = :herbig, differs = "Herbig; spiral arms, companion"),
    _N("HD 104237", 107.0; group = :herbig, differs = "Herbig; compact disk"),
    _N("HD 139614", 135.0; group = :herbig, differs = "Herbig; ring and gap"),
    _N("HD 141569", 111.0; group = :herbig, differs = "Herbig; gas-poor, nearly debris-like rings"),
    _N("HD 142666", 148.0; group = :herbig),
    _N("HD 145718", 152.0; group = :herbig),
    _N("HD 34282", 309.0; group = :herbig, differs = "Herbig; more distant, large cavity"),
    _N("V1247 Ori", 400.0; group = :herbig, differs = "Herbig; distant, ring + spiral"),
    _N("V892 Tau", 134.0; group = :herbig, differs = "Herbig Ae binary; circumbinary disk", simbad = "V892 Tau|Elias 1", aliases = ["Elias 1"]),
    _N("Elias 29", 137.0; group = :young, simbad = "Elias 29|[EC92] 29", aliases = ["Elias 2-29"]),
    _N("GY 91", 140.0; group = :young, differs = "Class I; embedded (eDisk)", simbad = "GY 91|[GY92] 91"),
    _N("IRAS 04302+2247", 160.0; group = :young, differs = "Edge-ish Class I; 'butterfly' nebula", aliases = ["IRAS04302"]),
    _N("IRAS 16253-2429", 140.0; group = :young, differs = "Class 0; embedded (eDisk)"),
    _N("IRS 63", 144.0; group = :young, differs = "Class I; embedded, rings (eDisk)", simbad = "IRS 63|Oph IRS 63|[GY92] 197", aliases = ["Oph IRS 63"]),
    _N("L1489 IRS", 140.0; group = :young, simbad = "IRAS 04016+2610|L1489 IRS", aliases = ["IRAS 04016+2610"]),
    _N("L1527 IRS", 140.0; group = :young, differs = "Class 0/I; near edge-on, embedded", simbad = "IRAS 04368+2557|L1527 IRS", aliases = ["IRAS 04368+2557", "L1527"]),
    _N("L1551 IRS 5", 140.0; group = :young, differs = "Class I binary; circumbinary disk"),
    _N("TMC1A", 140.0; group = :young, differs = "Class I; embedded", simbad = "IRAS 04365+2535|TMC1A", aliases = ["IRAS 04365+2535"]),
    _N("WL 17", 137.0; group = :young, differs = "Class I; ring + cavity, embedded", simbad = "WL 17|[WL78] 17"),
    _N("Flying Saucer", 138.0; group = :edge, simbad = "2MASS J16281370-2431391", aliases = ["2MASS J16281370-2431391"]),
    _N("HH 30", 140.0; group = :edge, differs = "Edge-on; jet-driving"),
    _N("HK Tau", 133.0; group = :edge, differs = "Binary; B disk edge-on, misaligned"),
    _N("Oph 163131", 147.0; group = :edge, differs = "Edge-on; large, gas-rich", simbad = "2MASS J16313124-2426281", aliases = ["2MASS J16313124-2426281"]),
    _N("Sz 133", 153.0; group = :edge),
    _N("EX Lup", 155.0; group = :outburst, differs = "EXor prototype; accretion-burst disk"),
    _N("FU Ori", 408.0; group = :outburst, differs = "FU Ori prototype; distant, bright"),
    _N("V883 Ori", 388.0; group = :outburst, differs = "FU Ori-type; water snowline resolved"),
    _N("V960 Mon", 1100.0; group = :outburst, differs = "FU Ori-type; very distant, fragmenting"),
]
sort!(ALMA_TARGETS; by = t -> _sortkey(t.name))

_fmt(x) = x === nothing ? "?" : @sprintf("%.1f", x)

"""
    print_target_menu(targets=ALMA_TARGETS; group=nothing, by=:name)

Print the menu: alphabetical by default (`by = :name`), or grouped by how TW Hya-like
a target is (`by = :group`). `group = :analogue` (or :cavity, :young, :herbig, :asym,
:edge, :outburst) shows one group only. Menu numbers are always the global numbers
(alphabetical), so they do not change with `by` or `group`.
"""
function print_target_menu(targets = ALMA_TARGETS; group = nothing, by::Symbol = :name)
    w = 124
    order = collect(eachindex(targets))
    if by === :group
        gi(t) = something(findfirst(==(t.group), GROUP_ORDER), length(GROUP_ORDER) + 1)
        sort!(order; by = k -> (gi(targets[k]), k))
    end
    println("="^w)
    println("ALMA target menu ($(length(targets)) disks), ",
            by === :group ? "grouped by how TW Hya-like they are" : "alphabetical")
    println("="^w)
    @printf("%3s  %-17s %-17s %6s %6s %7s  %s\n", "#", "Target", "Group", "d[pc]", "i[deg]", "PA[deg]",
            "How it differs from TW Hya")
    println("-"^w)
    prev = nothing
    for k in order
        t = targets[k]
        group !== nothing && t.group != group && continue
        by === :group && t.group != prev && (prev !== nothing && println("-"^w); prev = t.group)
        @printf("%3d  %-17s %-17s %6.1f %6s %7s  %s\n", k, t.name, get(GROUP_LABEL, t.group, string(t.group)),
                t.distance_pc, _fmt(t.incl_deg), _fmt(t.pa_deg), t.differs)
    end
    println("-"^w)
    println("Groups: T Tauri analogue = K-M stars, mostly axisymmetric | T Tauri + cavity = transition disks | Young/embedded = envelope")
    println("remnants | Herbig = A/B/F stars | Non-axisymmetric = horseshoes/arcs | Edge-on = cannot deproject | Outbursting = FU Ori/EXor.")
    println("d / i / PA are approximate literature values -- verify ('?' = unknown, treated face-on). Targets with '?' for both and no")
    println("coordinates in the catalogue are resolved by CDS Sesame when selected. print_target_menu(by = :group) groups the list.")
    println("Use custom_target(\"name\"; distance_pc=...) for anything not listed.")
    return nothing
end

"""
    resolve_name(name) -> (ra_deg, dec_deg)

Resolve an object name to ICRS coordinates with CDS Sesame (SIMBAD/NED/VizieR).
"""
function resolve_name(name::AbstractString)
    url = "https://cds.unistra.fr/cgi-bin/nph-sesame/-oI/A?" * HTTP.URIs.escapeuri(name)
    resp = HTTP.get(url; status_exception = false, read_idle_timeout = 30, retry = false)
    text = String(resp.body)
    m = match(r"^%J\s+([-+]?\d+\.?\d*)\s+([-+]?\d+\.?\d*)"m, text)
    m === nothing && error("Could not resolve \"$name\" via Sesame (HTTP $(resp.status)). " *
                           "Pass ra_deg= and dec_deg= explicitly.")
    return parse(Float64, m[1]), parse(Float64, m[2])
end

"""
    custom_target(name; ra_deg=nothing, dec_deg=nothing, distance_pc=nothing, incl_deg=nothing,
                  pa_deg=nothing, rmax_arcsec=2.0, proposal_id=nothing) -> AlmaTarget

Any ALMA target not in the menu. If `ra_deg`/`dec_deg` are omitted the name
is resolved with Sesame. Without `distance_pc`, radii in au assume 100 pc
(arcsec values are unaffected).
"""
function custom_target(name::AbstractString; ra_deg = nothing, dec_deg = nothing,
                       distance_pc = nothing, kwargs...)
    if ra_deg === nothing || dec_deg === nothing
        ra_deg, dec_deg = resolve_name(name)
        @printf("Resolved %s -> RA = %.5f deg, Dec = %.5f deg (CDS Sesame)\n", name, ra_deg, dec_deg)
    end
    known = distance_pc !== nothing
    return AlmaTarget(; name = String(name), keys = [_norm(name)], ra_deg = Float64(ra_deg),
                      dec_deg = Float64(dec_deg), distance_pc = known ? Float64(distance_pc) : 100.0,
                      distance_known = known, kwargs...)
end

"""
    _ensure_coords(t) -> AlmaTarget

Entries added without coordinates get RA/Dec from CDS Sesame (tries each name in
`t.simbad`, '|'-separated, then `t.name`). Entries that already have coordinates
are returned unchanged.
"""
function _ensure_coords(t::AlmaTarget)
    (isnan(t.ra_deg) || isnan(t.dec_deg)) || return t
    names = unique(String.(strip.(vcat(split(t.simbad, '|'), [t.name]))))
    filter!(!isempty, names)
    for nm in names
        try
            ra, dec = resolve_name(nm)
            @printf("Resolved %s (as \"%s\") -> RA = %.5f deg, Dec = %+.5f deg (CDS Sesame)\n", t.name, nm, ra, dec)
            vals = Any[getfield(t, f) for f in fieldnames(AlmaTarget)]
            vals[findfirst(==(:ra_deg), fieldnames(AlmaTarget))] = ra
            vals[findfirst(==(:dec_deg), fieldnames(AlmaTarget))] = dec
            return AlmaTarget(vals...)
        catch
        end
    end
    error("Could not resolve coordinates for \"$(t.name)\" (tried: $(join(names, ", "))). " *
          "Use custom_target(\"$(t.name)\"; ra_deg = ..., dec_deg = ..., distance_pc = $(t.distance_pc)).")
end

_default_index(targets) = something(findfirst(t -> t.name == "TW Hya", targets), 1)

"""
    select_target(choice=nothing) -> AlmaTarget

`choice` may be a menu number, a name (any spelling: "hd163296", "HD 163296"),
an `AlmaTarget` (e.g. from `custom_target`), or `nothing` to print the menu and
prompt for a choice (an input box appears in Jupyter/VS Code).
"""
function select_target(choice = nothing; targets = ALMA_TARGETS)
    if choice === nothing
        print_target_menu(targets)
        print("\nEnter a target number (1-$(length(targets))) or name [default $(_default_index(targets)) = $(targets[_default_index(targets)].name)]: ")
        flush(stdout)
        s = ""
        try
            s = strip(readline())
        catch
            println("(no input available; using the default)")
        end
        choice = isempty(s) ? _default_index(targets) : s
    end
    choice isa AlmaTarget && return _ensure_coords(choice)
    if choice isa Integer
        1 <= choice <= length(targets) || error("Target number must be between 1 and $(length(targets)).")
        return _ensure_coords(targets[choice])
    end
    s = String(choice)
    n = tryparse(Int, s)
    if n !== nothing
        1 <= n <= length(targets) || error("Target number must be between 1 and $(length(targets)).")
        return _ensure_coords(targets[n])
    end
    key = _norm(s)
    i = findfirst(t -> key in t.keys, targets)
    i === nothing && (i = findfirst(t -> any(k -> occursin(key, k), t.keys), targets))
    i === nothing && error("No menu target matches \"$s\". Use custom_target(\"$s\"; distance_pc=...).")
    return _ensure_coords(targets[i])
end

"""
    top_archive_targets(n=100; public_only=true, science=true, sort_by=:name, exclude=...) -> Vector{AlmaTarget}

OPTIONAL: rank target names by number of science rows (one per spectral window)
in the ALMA archive, via an ADQL GROUP BY on ivoa.obscore.
  * Name variants (HD_163296 / HD163296 / hd 163296 ...) are merged and their
    row counts summed.
  * `sort_by = :name` (default) lists the top n alphabetically; `:rows` lists by archive rows.
  * Names matching `exclude` (OFF-source pointings, the Sun by default) are dropped.
  * Names that match a curated catalogue entry (ALMA_TARGETS) return that entry, so
    distance, inclination, PA and literature features are kept. Others get a
    placeholder 100 pc -- pass distance_pc to run_alma_pipeline for those.
"""
function top_archive_targets(n::Integer = 100; public_only::Bool = true, science::Bool = true,
                             sort_by::Symbol = :name, exclude::Regex = r"(_OFF_?\d*$|^Sun(_\d+)?$)"i)
    conds = String[]
    science && push!(conds, "science_observation = 'T'")
    public_only && push!(conds, "data_rights = 'Public'")
    where_clause = isempty(conds) ? "" : "WHERE " * join(conds, " AND ") * " "
    fetch_n = max(10n, 300)      # over-fetch: merging and filtering shrink the list
    adql = "SELECT TOP $fetch_n target_name, COUNT(*) AS n_rows, AVG(s_ra) AS ra_deg, AVG(s_dec) AS dec_deg " *
           "FROM ivoa.obscore $(where_clause)GROUP BY target_name ORDER BY n_rows DESC"
    df = tap_query(adql)

    # merge name variants on the normalised key
    merged = Dict{String,NamedTuple}()
    order = String[]
    for row in eachrow(df)
        name = String(row.target_name)
        occursin(exclude, name) && continue
        cnt = tryparse(Int, string(row.n_rows)); cnt === nothing && continue
        ra, dec = tryparse(Float64, string(row.ra_deg)), tryparse(Float64, string(row.dec_deg))
        (ra === nothing || dec === nothing) && continue
        key = _norm(name)
        if haskey(merged, key)
            m = merged[key]
            merged[key] = (name = m.name, n = m.n + cnt, ra = m.ra, dec = m.dec)   # keep biggest variant's name/coords
        else
            merged[key] = (name = name, n = cnt, ra = ra, dec = dec)
            push!(order, key)
        end
    end
    ranked = sort(order; by = k -> -merged[k].n)[1:min(n, length(order))]   # the n most-observed
    # display order: :name = alphabetical (easy to scan), :rows = most-observed first
    sort_by === :name && sort!(ranked; by = k -> lowercase(merged[k].name))

    targets = AlmaTarget[]
    println("="^80)
    println("Top $n most-observed science targets in the ALMA archive, sorted by $(sort_by === :name ? "name" : "rows") (rows = spectral-window entries)")
    println("  name variants merged; OFF positions and the Sun excluded; * = curated catalogue entry")
    println("="^80)
    for key in ranked
        m = merged[key]
        j = findfirst(t -> key in t.keys, ALMA_TARGETS)
        if j !== nothing
            push!(targets, ALMA_TARGETS[j])
            @printf("%3d  %-28s rows = %-5d * %s, d = %.0f pc\n", length(targets), m.name, m.n,
                    ALMA_TARGETS[j].name, ALMA_TARGETS[j].distance_pc)
        else
            push!(targets, custom_target(m.name; ra_deg = m.ra, dec_deg = m.dec,
                                         note = "archive rows: $(m.n)"))
            @printf("%3d  %-28s rows = %d\n", length(targets), m.name, m.n)
        end
    end
    return targets
end


In [ ]:
print_target_menu()

## ALMA archive access (TAP query, datalink, mirror retry)

In [ ]:
# ---------------------------------------------------------------------------
# HTTP helpers: mirror rotation + retry/backoff, TAP queries, datalink
# ---------------------------------------------------------------------------

const TRANSIENT_STATUS = (502, 503, 504)

"""
    robust_get(path; query, max_attempts=3, base_delay=5) -> (response, mirror)

GET `<mirror><path>` with retry/backoff on transient errors (502/503/504,
network exceptions), rotating through `ALMA_MIRRORS`. Non-transient
responses (including 4xx) are returned to the caller to interpret.
"""
function robust_get(path::AbstractString; query = Dict{String,String}(),
                    max_attempts::Int = 3, base_delay::Real = 5)
    last_err = nothing
    for mirror in ALMA_MIRRORS
        for attempt in 1:max_attempts
            println("  [mirror=$mirror, attempt $attempt/$max_attempts] GET $path ...")
            try
                resp = HTTP.get(mirror * path; query = query, status_exception = false,
                                retry = false, read_idle_timeout = 180, connect_timeout = 30)
                if resp.status in TRANSIENT_STATUS
                    last_err = "HTTP $(resp.status)"
                    if attempt < max_attempts
                        delay = base_delay * attempt
                        println("  Transient error ($(last_err)); retrying in $(delay)s ...")
                        sleep(delay)
                        continue
                    end
                    break
                end
                return resp, mirror
            catch err
                last_err = sprint(showerror, err)
                if attempt < max_attempts
                    delay = base_delay * attempt
                    println("  Transient error ($(typeof(err))); retrying in $(delay)s ...")
                    sleep(delay)
                end
            end
        end
        println("  Mirror $mirror did not succeed, trying next mirror ...")
    end
    error("GET $path failed across all mirrors $(ALMA_MIRRORS). Last error: $last_err")
end

# ---- minimal VOTable (TABLEDATA) reader -----------------------------------

"""
    parse_votable(text) -> DataFrame

Parse the first TABLE of a VOTable using the TABLEDATA serialization into an
all-`String` DataFrame. Raises on a server-reported error or on BINARY/BINARY2
serializations (request CSV or a TABLEDATA VOTable instead).
"""
function parse_votable(text::AbstractString)
    # Strip XML namespace declarations so plain XPath works.
    clean = replace(text, r"\s(xmlns(:\w+)?|xsi:\w+)=\"[^\"]*\"" => "")
    doc = EzXML.parsexml(clean)

    err_node = findfirst("//INFO[@name='QUERY_STATUS'][@value='ERROR']", doc)
    if err_node !== nothing
        error("Server reported a query error: ", strip(nodecontent(err_node)))
    end

    table = findfirst("//RESOURCE[@type='results']/TABLE", doc)
    table === nothing && (table = findfirst("//TABLE", doc))
    table === nothing && error("No <TABLE> found in VOTable response.")

    if findfirst("./DATA/BINARY", table) !== nothing || findfirst("./DATA/BINARY2", table) !== nothing
        error("VOTable uses BINARY serialization; only TABLEDATA/CSV are supported here.")
    end

    names = [field["name"] for field in findall("./FIELD", table)]
    rows = findall("./DATA/TABLEDATA/TR", table)
    cols = [String[] for _ in names]
    for tr in rows
        tds = findall("./TD", tr)
        for (k, col) in enumerate(cols)
            push!(col, k <= length(tds) ? String(strip(nodecontent(tds[k]))) : "")
        end
    end
    return DataFrame([Symbol(n) => c for (n, c) in zip(names, cols)])
end

"""
    tap_query(adql) -> DataFrame

Run a synchronous ADQL query against the ALMA TAP service (mirror rotation
and retry included). Requests a VOTable first (verified against the ALMA
service) and falls back to CSV. All columns are returned as `String`.
"""
function tap_query(adql::AbstractString)
    last_msg = ""
    for fmt in ("votable", "csv")
        query = Dict("REQUEST" => "doQuery", "LANG" => "ADQL",
                     "FORMAT" => fmt, "QUERY" => adql)
        resp, mirror = robust_get("/tap/sync"; query = query)
        text = String(resp.body)
        _dbg("tap_query: format =", fmt, "| HTTP status =", resp.status,
             "| bytes =", length(text), "| mirror =", mirror)
        try
            resp.status == 200 || error("HTTP $(resp.status): ", first(text, 400))
            if startswith(lstrip(text), "<")
                return parse_votable(text)
            else
                df = CSV.read(IOBuffer(text), DataFrame; types = String)
                return mapcols(c -> coalesce.(c, ""), df)
            end
        catch err
            last_msg = sprint(showerror, err)
            println("  FORMAT=$fmt failed: ", first(last_msg, 300))
        end
    end
    error("TAP query failed in both CSV and VOTable formats. Last error: $last_msg")
end

"""
    robust_get_data_info(member_ous_uid) -> DataFrame

Datalink listing of the downloadable products for one member OUS (the
equivalent of astroquery's `Alma.get_data_info`). Returns the datalink
table with empty-URL rows removed and URLs percent-decoded.

The ALMA datalink service only exposes whole `.tar` packages for this kind
of delivery, so astroquery's `expand_tarfiles=True` returns the same
listing and no separate option is needed here.
"""
function robust_get_data_info(member_ous_uid::AbstractString)
    println("  fetching data info for $member_ous_uid ...")
    resp, mirror = robust_get("/datalink/sync"; query = Dict("ID" => String(member_ous_uid)))
    resp.status == 200 || error("datalink returned HTTP $(resp.status): ", first(String(resp.body), 400))
    df = parse_votable(String(resp.body))
    _dbg("robust_get_data_info: columns =", names(df), "| rows =", nrow(df), "| mirror =", mirror)
    "access_url" in names(df) || error("datalink response has no access_url column; columns = $(names(df))")
    df = df[.!isempty.(df.access_url), :]
    df.access_url = [HTTP.URIs.unescapeuri(u) for u in df.access_url]
    return df
end

## Query → choose observation → inspect → pick product tarball

In [ ]:
# ---------------------------------------------------------------------------
# Query -> choose observation -> inspect -> pick product tarball
# ---------------------------------------------------------------------------

_col(df, name) = name in names(df) ? df[!, name] : fill("", nrow(df))

"""
    query_target(t::AlmaTarget; public_only=true, science=true, proposal_id=t.proposal_id,
                 radius_deg=0.05, name_filter=true) -> DataFrame

Query ivoa.obscore with a cone around the target position (server side), then
keep rows whose `target_name` matches one of the target's name aliases
(spaces/underscores/case ignored). `name_filter=false` keeps everything the cone
returns. `proposal_id` restricts to one project.
"""
function query_target(t::AlmaTarget; public_only::Bool = true, science::Bool = true,
                      proposal_id = t.proposal_id, radius_deg::Real = 0.05, name_filter::Bool = true)
    conds = ["INTERSECTS(CIRCLE('ICRS', $(t.ra_deg), $(t.dec_deg), $radius_deg), s_region) = 1"]
    public_only && push!(conds, "data_rights = 'Public'")
    science && push!(conds, "science_observation = 'T'")
    proposal_id === nothing || push!(conds, "proposal_id = '$(strip(proposal_id))'")

    select = "SELECT obs_id, target_name, member_ous_uid, band_list, frequency, " *
             "spatial_resolution, velocity_resolution, t_exptime, proposal_id, " *
             "obs_release_date, science_keyword FROM ivoa.obscore WHERE "
    println("Querying ALMA archive for $(t.name) ...")
    _dbg("query_target: ADQL =", select * join(conds, " AND "))

    results = try
        tap_query(select * join(conds, " AND "))
    catch err
        println("Query with public/science filters failed (", first(sprint(showerror, err), 200),
                "); retrying with position", proposal_id === nothing ? "" : " + proposal", " only ...")
        base = [conds[1]]
        proposal_id === nothing || push!(base, last(conds))
        tap_query(select * join(base, " AND "))
    end
    println("Found $(nrow(results)) rows in the cone.")

    if name_filter && nrow(results) > 0
        name_col = "target_name" in names(results) ? :target_name : :obs_id
        keep = [any(k -> occursin(k, _norm(string(v))), t.keys) for v in results[!, name_col]]
        if !any(keep)
            seen = unique(string.(results[!, name_col]))
            println("No target_name matched $(t.keys). Distinct names in the cone: ",
                    join(first(seen, 10), ", "), length(seen) > 10 ? ", ..." : "")
            println("Re-run with name_filter=false to keep all rows in the cone.")
        end
        results = results[keep, :]
        println("Filtered to $(nrow(results)) rows matching $(t.name).")
    end
    if nrow(results) == 0
        println("No rows. Try public_only=false (proprietary data shows up but can't be downloaded), " *
                "a larger radius_deg, or proposal_id=nothing.")
    end
    return results
end

"""
    summarize_observations(results) -> DataFrame

One row per member OUS (an observing block set; several spectral-window rows
collapse into one), sorted by best (finest) spatial resolution in arcsec.
"""
function summarize_observations(results::DataFrame)
    uids = String.(_col(results, "member_ous_uid"))
    res  = String.(_col(results, "spatial_resolution"))
    band = String.(_col(results, "band_list"))
    prop = String.(_col(results, "proposal_id"))
    texp = String.(_col(results, "t_exptime"))
    rel  = String.(_col(results, "obs_release_date"))
    rows = NamedTuple[]
    for u in unique(uids)
        idx = findall(==(u), uids)
        rv = Float64[x for x in (tryparse(Float64, res[i]) for i in idx) if x !== nothing]
        tv = Float64[x for x in (tryparse(Float64, texp[i]) for i in idx) if x !== nothing]
        push!(rows, (member_ous_uid = u, proposal_id = prop[idx[1]], band = band[idx[1]],
                     best_res_arcsec = isempty(rv) ? NaN : minimum(rv),
                     t_exp_s = isempty(tv) ? NaN : maximum(tv),
                     release = first(rel[idx[1]], 10), n_rows = length(idx)))
    end
    df = DataFrame(rows)
    nrow(df) > 0 && sort!(df, :best_res_arcsec)        # NaN sorts last
    return df
end

"""
    select_observation(obs; band=nothing, choice=1) -> Union{DataFrameRow,Nothing}

Print the (up to 15) finest-resolution observations, optionally restricted to
a band (e.g. `band=7`), and return row number `choice` of that list.
"""
function select_observation(obs::DataFrame; band = nothing, choice::Integer = 1)
    sel = obs
    if band !== nothing
        sel = obs[[occursin(string(band), b) for b in obs.band], :]
        println("Restricted to band $band: $(nrow(sel)) of $(nrow(obs)) observations.")
    end
    if nrow(sel) == 0
        println("No observations left after filtering.")
        return nothing
    end
    println("\nObservations (member OUS), finest spatial resolution first:")
    show(stdout, MIME("text/plain"), first(sel, 15); allcols = true, allrows = true, truncate = 40)
    println()
    1 <= choice <= nrow(sel) || error("obs_choice must be between 1 and $(nrow(sel)).")
    row = sel[choice, :]
    @printf("\nUsing observation #%d: %s (proposal %s, band %s, best resolution %.4f\")\n",
            choice, row.member_ous_uid, row.proposal_id, row.band, row.best_res_arcsec)
    return row
end

function _to_gb(s)
    x = tryparse(Float64, s)
    return x === nothing ? missing : round(x / 1e9; digits = 3)
end

"""
    inspect_products(member_ous_uid) -> DataFrame

List the downloadable packages (tar files) for one member OUS, with sizes in GB.
"""
function inspect_products(member_ous_uid::AbstractString)
    println("\nFetching file listing for $member_ous_uid ...")
    data_info = robust_get_data_info(member_ous_uid)
    if "content_length" in names(data_info)
        data_info.size_GB = [_to_gb(s) for s in data_info.content_length]
    end
    cols = filter(c -> c in names(data_info), ["access_url", "content_type", "size_GB"])
    show(stdout, MIME("text/plain"), data_info[:, cols]; allcols = true, allrows = true, truncate = 110)
    println()
    return data_info
end

"""
    pick_product_tarball(data_info) -> Union{String,Nothing}

ALMA deliveries often expose only whole `.tar` packages. Preference: (1) the
plain main product tar (pipeline images), (2) the `external_ari_l` tar
(reprocessed images). Raw ASDM tars and the auxiliary tar are skipped.
"""
function pick_product_tarball(data_info::DataFrame)
    urls = String.(data_info.access_url)

    is_main_product_tar(u) = let lu = lowercase(u)
        endswith(lu, ".tar") && !occursin("auxiliary", lu) && !occursin("asdm", lu) &&
            !occursin("external_ari_l", lu) && !occursin("readme", lu)
    end
    is_ari_l_tar(u) = let lu = lowercase(u)
        endswith(lu, ".tar") && occursin("external_ari_l", lu)
    end

    main_candidates = filter(is_main_product_tar, urls)
    isempty(main_candidates) || return first(main_candidates)

    ari_l_candidates = filter(is_ari_l_tar, urls)
    if !isempty(ari_l_candidates)
        println("No plain main-product tar found; falling back to the external_ari_l (reprocessed) tarball.")
        return first(ari_l_candidates)
    end
    println("No suitable product tarball found -- only auxiliary/raw ASDM tars are available for this UID.")
    return nothing
end

"Size in GB of `url` according to the datalink listing, or `nothing`."
function tarball_size_gb(data_info::DataFrame, url::AbstractString)
    "content_length" in names(data_info) || return nothing
    i = findfirst(==(url), String.(data_info.access_url))
    i === nothing && return nothing
    x = tryparse(Float64, String(data_info.content_length[i]))
    return x === nothing ? nothing : x / 1e9
end

# Candidate FITS ordering: continuum + pbcor first; PSFs, residuals, models,
# masks, primary-beam maps, cubes and moment maps last.
const _JUNK_TAGS = ("psf", "residual", "model", "mask", "flux", "alpha", "sigma", "rms", "cube", "mom", ".pb.", "_pb.")

function _fits_score(f)
    lf = lowercase(basename(f))
    s = occursin("cont", lf) && occursin("pbcor", lf) ? 0 :
        occursin("cont", lf) ? 1 : occursin("pbcor", lf) ? 2 : 3
    return any(j -> occursin(j, lf), _JUNK_TAGS) ? s + 10 : s
end

rank_fits_files(files) = sort(files; by = _fits_score, alg = MergeSort)

## Download tarball and extract FITS members

In [ ]:
# ---------------------------------------------------------------------------
# Download a product tarball and extract only the FITS members
# ---------------------------------------------------------------------------

_find_fits(dir; pattern = r"\.fits$"i) =
    isdir(dir) ? [joinpath(root, f) for (root, _, files) in walkdir(dir)
                  for f in files if occursin(pattern, f)] : String[]

"""
    download_and_extract_fits(tarball_url; dest_dir=CACHE_DIR, fits_regex, delete_tar=true) -> Vector{String}

Download a single ALMA product tarball and extract only the FITS members
from it, then delete the tarball (ALMA only exposes whole `.tar` packages,
which can be multiple GB, so we don't keep them). Skips everything if FITS
files were already extracted on a previous run.
"""
function download_and_extract_fits(tarball_url::AbstractString; dest_dir::AbstractString = CACHE_DIR,
                                   fits_regex::Regex = r"\.fits$"i, delete_tar::Bool = true)
    tarball_name = basename(tarball_url)
    tar_local = joinpath(dest_dir, tarball_name)
    extract_dir = joinpath(dest_dir, splitext(tarball_name)[1] * "_extracted")

    existing = _find_fits(extract_dir; pattern = fits_regex)
    if !isempty(existing)
        println("Already have $(length(existing)) extracted FITS file(s), skipping download.")
        return existing
    end

    if !isfile(tar_local)
        println("\nDownloading $tarball_url ...")
        part = tar_local * ".part"          # never leave a truncated .tar that looks complete
        last_pct = Ref(-10)
        Downloads.download(tarball_url, part; progress = (total, now) -> begin
            total > 0 || return
            pct = floor(Int, 100 * now / total)
            if pct >= last_pct[] + 10
                last_pct[] = pct
                @printf("  %3d%%  (%.2f / %.2f GB)\n", pct, now / 1e9, total / 1e9)
            end
        end)
        mv(part, tar_local; force = true)
        println("Download complete.")
    else
        println("Found cached tarball at $tar_local.")
    end

    println("Extracting FITS members ...")
    headers = Tar.list(tar_local; strict = false)
    members = String[h.path for h in headers if h.type == :file && occursin(fits_regex, h.path)]
    if isempty(members)
        println("No FITS members found inside this tarball.")
    else
        wanted = Set(members)
        isdir(extract_dir) && rm(extract_dir; recursive = true, force = true)   # Tar.extract needs an empty target
        try
            Tar.extract(h -> h.path in wanted, tar_local, extract_dir)
        catch err
            println("Tar.extract failed: ", sprint(showerror, err))
        end
        if !all(m -> isfile(joinpath(extract_dir, m)), members)
            println("Pure-Julia extraction incomplete; falling back to system `tar` ...")
            mkpath(extract_dir)
            run(`tar -xf $tar_local -C $extract_dir $members`)
        end
    end
    extracted_paths = [joinpath(extract_dir, m) for m in members if isfile(joinpath(extract_dir, m))]

    # Only delete the (large) tarball once we actually have the FITS files.
    if delete_tar && !isempty(extracted_paths) && isfile(tar_local)
        rm(tar_local)
        println("Deleted $tarball_name after extraction to save space.")
    end
    return extracted_paths
end

## FITS loading and image display

In [ ]:
# ---------------------------------------------------------------------------
# FITS loading + 2D image display
# ---------------------------------------------------------------------------

_hget(h, key, default = nothing) = haskey(h, key) ? h[key] : default

"""
    pixel_scales_arcsec(header) -> (cdelt1_arcsec, cdelt2_arcsec)

Signed pixel scales along the two sky axes, from CDELT1/2 or the CD matrix.
RA normally increases to the left, so `cdelt1` is negative.
"""
function pixel_scales_arcsec(h)
    if haskey(h, "CDELT1") && haskey(h, "CDELT2")
        return 3600.0 * h["CDELT1"], 3600.0 * h["CDELT2"]
    elseif haskey(h, "CD1_1")
        cd11, cd12 = h["CD1_1"], _hget(h, "CD1_2", 0.0)
        cd21, cd22 = _hget(h, "CD2_1", 0.0), h["CD2_2"]
        s1, s2 = hypot(cd11, cd21), hypot(cd12, cd22)
        flip = (cd11 * cd22 - cd12 * cd21) < 0
        return 3600.0 * (flip ? -s1 : s1), 3600.0 * s2
    else
        error("FITS header has neither CDELT1/2 nor a CD matrix; can't determine pixel scale.")
    end
end

"""
    load_fits_image(filepath) -> NamedTuple

Read the first image HDU with >= 2 axes, collapse trailing axes
(frequency/Stokes) to their first plane (with a warning if they were longer
than 1, i.e. a cube), and return the 2D array `[x, y]` plus header values.
"""
function load_fits_image(filepath::AbstractString)
    f = FITS(filepath)
    try
        idx = findfirst(i -> f[i] isa ImageHDU && ndims(f[i]) >= 2, 1:length(f))
        idx === nothing && error("No image HDU with data found in $filepath")
        hdu = f[idx]
        raw = read(hdu)
        h = read_header(hdu)

        if ndims(raw) > 2 && any(>(1), size(raw)[3:end])
            println("WARNING: $(basename(filepath)) has image shape $(size(raw)); using only the first plane. ",
                    "This looks like a cube, not a continuum image.")
        end
        arr = raw
        while ndims(arr) > 2
            arr = selectdim(arr, ndims(arr), 1)
        end
        data = Array(arr)

        cdelt1, cdelt2 = pixel_scales_arcsec(h)
        return (
            path = String(filepath),
            data = data,
            cdelt1_arcsec = cdelt1,
            cdelt2_arcsec = cdelt2,
            pix_scale_arcsec = abs(cdelt1),
            bmaj_deg = _hget(h, "BMAJ"),
            bmin_deg = _hget(h, "BMIN"),
            bpa_deg  = _hget(h, "BPA"),
            bunit    = String(strip(string(_hget(h, "BUNIT", "Jy/beam")))),
        )
    finally
        close(f)
    end
end

"""
    display_fits_image(img; label="", center=nothing, cutout_arcsec=6.0, vmax_percentile=99.5,
                       asinh_a=0.01, filename="continuum_image.png") -> Figure

Display a 2D image with sky-offset axes (east to the left).

cutout_arcsec: square cutout size around `center` (1-based `(x, y)` pixel;
    default the image centre).
vmax_percentile: central percentile interval for the colour limits (astropy
    PercentileInterval semantics; 99.5 clips 0.25% at each end).
asinh_a: asinh stretch softening, y = asinh(x/a)/asinh(1/a) on the normalised
    image. Smaller values compress the bright core harder.
"""
function display_fits_image(img; label = "", center = nothing, cutout_arcsec = 6.0,
                            vmax_percentile = 99.5, asinh_a = 0.01, filename = "continuum_image.png")
    data = img.data
    nx, ny = size(data)
    cx, cy = center === nothing ? (nx ÷ 2 + 1, ny ÷ 2 + 1) : round.(Int, center)

    ilo, ihi, jlo, jhi = 1, nx, 1, ny
    if cutout_arcsec !== nothing
        size_pix = max(round(Int, cutout_arcsec / img.pix_scale_arcsec), 10)
        half = size_pix ÷ 2
        ilo = max(cx - half, 1);  ihi = min(ilo + size_pix - 1, nx)
        jlo = max(cy - half, 1);  jhi = min(jlo + size_pix - 1, ny)
    end
    cut = Float64.(data[ilo:ihi, jlo:jhi])

    vals = filter(isfinite, vec(cut))
    tail = (100 - vmax_percentile) / 200
    vmin, vmax = quantile(vals, [tail, 1 - tail])
    stretch(x) = asinh(x / asinh_a) / asinh(1 / asinh_a)
    normed = stretch.(clamp.((cut .- vmin) ./ (vmax - vmin), 0.0, 1.0))

    xs = range((ilo - cx) * img.cdelt1_arcsec, step = img.cdelt1_arcsec, length = size(cut, 1))
    ys = range((jlo - cy) * img.cdelt2_arcsec, step = img.cdelt2_arcsec, length = size(cut, 2))
    if img.cdelt1_arcsec < 0
        xs = reverse(xs);  normed = reverse(normed; dims = 1)
    end
    if img.cdelt2_arcsec < 0
        ys = reverse(ys);  normed = reverse(normed; dims = 2)
    end

    fig = Figure(size = (760, 700))
    ax = Axis(fig[1, 1]; aspect = DataAspect(), xreversed = true,
              xlabel = "ΔRA [arcsec]", ylabel = "ΔDec [arcsec]",
              title = "$(label) — $(basename(img.path))", titlesize = 12)
    hm = heatmap!(ax, xs, ys, normed; colormap = :inferno, colorrange = (0, 1))

    tick_vals = collect(range(vmin, vmax; length = 6))
    tick_pos = stretch.((tick_vals .- vmin) ./ (vmax - vmin))
    Colorbar(fig[1, 2], hm; label = "Flux density [$(img.bunit)]",
             ticks = (tick_pos, [@sprintf("%.1e", v) for v in tick_vals]))

    display(fig)
    _save_fig(fig, filename)
    return fig
end

## Disk geometry, centre fitting and radial profile

In [ ]:
# ---------------------------------------------------------------------------
# Disk geometry, centre fitting, deprojected radial profile
# ---------------------------------------------------------------------------

"""
Geometry tuple for `_dep_radius`. `pa_deg` is the position angle of the disk
major axis, east of north; `incl_deg` the inclination (0 = face-on).
"""
function _geometry(img, incl_deg, pa_deg)
    sin_pa, cos_pa = sincos(deg2rad(pa_deg))
    return (e = img.cdelt1_arcsec, n = img.cdelt2_arcsec,
            sin_pa = sin_pa, cos_pa = cos_pa, cos_i = cos(deg2rad(incl_deg)))
end

# Deprojected (face-on) radius in arcsec of pixel (i, j) about centre (cx, cy).
# Sky offsets: east = (i - cx) * cdelt1, north = (j - cy) * cdelt2. The major
# axis is at PA east of north, i.e. along (sin PA, cos PA) in (east, north); the
# minor-axis offset is foreshortened by cos(i) and is stretched back here.
@inline function _dep_radius(i, j, cx, cy, g)
    E = (i - cx) * g.e
    N = (j - cy) * g.n
    xmaj = E * g.sin_pa + N * g.cos_pa
    xmin = E * g.cos_pa - N * g.sin_pa
    return sqrt(xmaj * xmaj + (xmin / g.cos_i)^2)
end

"""
    fit_center(img, g; start, search_arcsec=0.15, fit_radius_arcsec=1.0, nb=50) -> (cx, cy)

Refine the disk centre (1-based pixel) by minimising the total within-annulus
sum of squares of the image inside `fit_radius_arcsec` -- the centre that makes
the disk most axisymmetric. Coarse grid search, then a 1-pixel refinement.
Works for ring/cavity disks where the brightest pixel is NOT the star.
"""
function fit_center(img, g; start, search_arcsec = 0.15, fit_radius_arcsec = 1.0, nb::Int = 50)
    data = img.data
    nx, ny = size(data)
    pix = img.pix_scale_arcsec
    rpix = ceil(Int, fit_radius_arcsec / pix)

    function objective(cx, cy)
        ilo = max(1, cx - rpix); ihi = min(nx, cx + rpix)
        jlo = max(1, cy - rpix); jhi = min(ny, cy + rpix)
        sums = zeros(Float64, nb); sumsq = zeros(Float64, nb); cnt = zeros(Int, nb)
        @inbounds for j in jlo:jhi, i in ilo:ihi
            v = data[i, j]
            isnan(v) && continue
            r = _dep_radius(i, j, cx, cy, g)
            r < fit_radius_arcsec || continue
            b = floor(Int, r / fit_radius_arcsec * nb) + 1
            x = Float64(v)
            sums[b] += x; sumsq[b] += x * x; cnt[b] += 1
        end
        ss = 0.0
        for b in 1:nb
            cnt[b] > 1 && (ss += sumsq[b] - sums[b]^2 / cnt[b])
        end
        return ss
    end

    function search(c0, R, step)
        best = (Inf, c0[1], c0[2])
        for cx in (c0[1] - R):step:(c0[1] + R), cy in (c0[2] - R):step:(c0[2] + R)
            (1 <= cx <= nx && 1 <= cy <= ny) || continue
            o = objective(cx, cy)
            o < best[1] && (best = (o, cx, cy))
        end
        return best
    end

    R = max(1, round(Int, search_arcsec / pix))
    step = max(1, R ÷ 8)
    b1 = search(start, R, step)
    b2 = step > 1 ? search((b1[2], b1[3]), step, 1) : b1
    _dbg("fit_center: start =", start, "coarse =", (b1[2], b1[3]), "fine =", (b2[2], b2[3]))
    return (b2[2], b2[3])
end

"""
    extract_radial_profile(img; incl_deg=0.0, pa_deg=0.0, distance_pc=100.0, max_radius_arcsec=2.0,
                           n_bins=nothing, center=nothing, center_mode=:optimize) -> NamedTuple

Azimuthally averaged, deprojected radial profile.

center_mode (used when `center` is `nothing`):
  :optimize     refine the image centre by symmetry (default; see `fit_center`)
  :image_center the image's central pixel (ALMA images are phase-centred on the target)
  :peak         the brightest pixel (only sensible for centrally peaked disks)
n_bins: number of radial annuli; `nothing` picks a bin width of half the beam
    major axis (40-400 bins). Adjacent bins are correlated at that sampling.
"""
function extract_radial_profile(img; incl_deg = 0.0, pa_deg = 0.0, distance_pc = 100.0,
                                max_radius_arcsec = 2.0, n_bins = nothing,
                                center = nothing, center_mode::Symbol = :optimize)
    data = img.data
    nx, ny = size(data)
    pix = img.pix_scale_arcsec
    _dbg("extract_radial_profile: image", img.path)
    _dbg("extract_radial_profile: data shape", size(data), "eltype", eltype(data),
         "nan count", count(isnan, data), "finite count", count(isfinite, data))

    bmaj_arcsec = bmin_arcsec = bmaj_au = bmin_au = nothing
    if img.bmaj_deg !== nothing && img.bmin_deg !== nothing
        bmaj_arcsec = img.bmaj_deg * 3600.0
        bmin_arcsec = img.bmin_deg * 3600.0
        bmaj_au = bmaj_arcsec * distance_pc
        bmin_au = bmin_arcsec * distance_pc
        @printf("Synthesized beam: %.4f\" x %.4f\" (PA=%s) = %.2f au x %.2f au at %.1f pc\n",
                bmaj_arcsec, bmin_arcsec, img.bpa_deg, bmaj_au, bmin_au, distance_pc)
    else
        println("WARNING: no BMAJ/BMIN in the FITS header -- beam size unknown.")
    end

    # Keep the profile inside the image.
    rimg = 0.5 * min(nx, ny) * pix
    if max_radius_arcsec > 0.95 * rimg
        println(@sprintf("max_radius_arcsec = %.2f exceeds the image half-width (%.2f\"); reducing to %.2f\".",
                         max_radius_arcsec, rimg, 0.95 * rimg))
        max_radius_arcsec = 0.95 * rimg
    end

    if n_bins === nothing
        binw = bmaj_arcsec === nothing ? 10 * pix : 0.5 * bmaj_arcsec
        n_bins = clamp(round(Int, max_radius_arcsec / binw), 40, 400)
    end
    n_bins < typemax(Int16) || error("n_bins too large")

    g = _geometry(img, incl_deg, pa_deg)
    c_img = (nx ÷ 2 + 1, ny ÷ 2 + 1)
    if center !== nothing
        cx, cy = round.(Int, center)
    elseif center_mode == :peak
        _, ci = findmax(v -> isnan(v) ? -Inf : v, data)
        cx, cy = Tuple(ci)
    elseif center_mode == :image_center
        cx, cy = c_img
    elseif center_mode == :optimize
        cx, cy = fit_center(img, g; start = c_img,
                            fit_radius_arcsec = min(1.0, 0.8 * max_radius_arcsec, 0.9 * rimg))
    else
        error("center_mode must be :optimize, :image_center or :peak")
    end
    @printf("Centre pixel (1-based): (%d, %d); offset from image centre: dRA = %+.4f\", dDec = %+.4f\"\n",
            cx, cy, (cx - c_img[1]) * img.cdelt1_arcsec, (cy - c_img[2]) * img.cdelt2_arcsec)

    binw = max_radius_arcsec / n_bins
    binidx = zeros(Int16, nx, ny)
    sums   = zeros(Float64, n_bins)
    counts = zeros(Int, n_bins)
    @inbounds for j in 1:ny, i in 1:nx
        v = data[i, j]
        isnan(v) && continue
        r = _dep_radius(i, j, cx, cy, g)
        r < max_radius_arcsec || continue
        b = floor(Int, r / binw) + 1
        b > n_bins && continue
        binidx[i, j] = b
        sums[b] += v
        counts[b] += 1
    end

    mean_flux = [counts[b] > 0 ? sums[b] / counts[b] : NaN for b in 1:n_bins]
    ssq = zeros(Float64, n_bins)
    @inbounds for j in 1:ny, i in 1:nx
        b = binidx[i, j]
        b == 0 && continue
        d = data[i, j] - mean_flux[b]
        ssq[b] += d * d
    end
    std_flux = [counts[b] > 0 ? sqrt(ssq[b] / counts[b]) : NaN for b in 1:n_bins]   # population std

    bin_edges = range(0, max_radius_arcsec; length = n_bins + 1)
    bin_centers = [0.5 * (bin_edges[k] + bin_edges[k + 1]) for k in 1:n_bins]
    @printf("Radial profile: %d bins of %.4f\" (%.2f au) out to %.2f\" (%.0f au)\n",
            n_bins, binw, binw * distance_pc, max_radius_arcsec, max_radius_arcsec * distance_pc)
    _dbg("extract_radial_profile: empty bins =", count(==(0), counts),
         "valid mean_flux entries =", count(!isnan, mean_flux))
    all(isnan, mean_flux) && println("WARNING: every radial bin is NaN -- check the centre, pixel scale and radius.")

    return (
        radius_arcsec = bin_centers, radius_au = bin_centers .* distance_pc,
        mean_flux = mean_flux, std_flux = std_flux,
        pix_scale_arcsec = pix, center_pix = (cx, cy), incl_deg = incl_deg, pa_deg = pa_deg,
        bmaj_arcsec = bmaj_arcsec, bmin_arcsec = bmin_arcsec, bmaj_au = bmaj_au, bmin_au = bmin_au,
        distance_pc = distance_pc, bunit = img.bunit,
    )
end

## Peak finding and ring/gap detection

In [ ]:
# ---------------------------------------------------------------------------
# Signal-processing helpers (ports of scipy.signal.savgol_filter / find_peaks)
# ---------------------------------------------------------------------------

"""
    savgol_filter(y, window, polyorder) -> Vector{Float64}

Savitzky-Golay smoothing equivalent to `scipy.signal.savgol_filter(y,
window, polyorder)` with its default `mode="interp"`: interior points use
the least-squares polynomial fit over the centred window; the first and last
`window÷2` points evaluate the polynomial fitted to the first/last `window`
samples.
"""
function savgol_filter(y::AbstractVector{<:Real}, window::Integer, polyorder::Integer)
    isodd(window) || throw(ArgumentError("window must be odd, got $window"))
    window > polyorder || throw(ArgumentError("window must exceed polyorder"))
    n = length(y)
    window <= n || throw(ArgumentError("window ($window) larger than data length ($n)"))
    h = window ÷ 2
    xs = collect(-h:h)
    V = [Float64(x)^k for x in xs, k in 0:polyorder]    # window × (polyorder+1)
    P = pinv(V)                                         # least-squares solver
    out = zeros(Float64, n)
    for i in 1:n
        s = clamp(i - h, 1, n - window + 1)            # window start (shifted at the edges)
        c = P * Float64.(y[s:(s + window - 1)])
        xi = Float64(i - (s + h))                       # position of i relative to the window centre
        out[i] = sum(c[k + 1] * xi^k for k in 0:polyorder)
    end
    return out
end

# Local maxima, handling flat plateaus by taking the middle sample
# (scipy.signal._peak_finding_utils._local_maxima_1d).
function _local_maxima(x::AbstractVector)
    n = length(x)
    mids = Int[]
    n < 3 && return mids
    i = 2
    i_max = n
    while i < i_max
        if x[i - 1] < x[i]
            i_ahead = i + 1
            while i_ahead < i_max && x[i_ahead] == x[i]
                i_ahead += 1
            end
            if x[i_ahead] < x[i]
                push!(mids, (i + (i_ahead - 1)) ÷ 2)
                i = i_ahead
            end
        end
        i += 1
    end
    return mids
end

# Topographic prominence of the peak at index p with no window limit
# (scipy.signal._peak_finding_utils._peak_prominences, wlen=None).
function _peak_prominence(x::AbstractVector, p::Int)
    n = length(x)
    left_min = x[p]
    i = p
    while i >= 1 && x[i] <= x[p]
        x[i] < left_min && (left_min = x[i])
        i -= 1
    end
    right_min = x[p]
    i = p
    while i <= n && x[i] <= x[p]
        x[i] < right_min && (right_min = x[i])
        i += 1
    end
    return x[p] - max(left_min, right_min)
end

"""
    find_peaks(x; prominence=nothing) -> (indices, prominences)

Indices (1-based) of local maxima of `x` whose prominence is at least
`prominence`, with their prominences -- the subset of
`scipy.signal.find_peaks` used by the original script.
"""
function find_peaks(x::AbstractVector{<:Real}; prominence = nothing)
    peaks = _local_maxima(x)
    proms = [_peak_prominence(x, p) for p in peaks]
    if prominence !== nothing
        keep = proms .>= prominence
        peaks, proms = peaks[keep], proms[keep]
    end
    return peaks, proms
end

# ---------------------------------------------------------------------------
# Ring / gap detection
# ---------------------------------------------------------------------------

"""
    find_rings_and_gaps(profile; ring_prominence=nothing, gap_prominence=nothing) -> (rings, gaps)

Locate local maxima (rings) and minima (gaps) in a radial flux profile with
`find_peaks`. `prominence=nothing` derives a threshold from the profile's own
scatter (3x the median `std_flux`); pass an explicit value to override if the
auto threshold is too strict/loose.
"""
function find_rings_and_gaps(profile; ring_prominence = nothing, gap_prominence = nothing)
    valid = .!isnan.(profile.mean_flux)
    flux_v = profile.mean_flux[valid]
    radius_v = profile.radius_au[valid]
    _dbg("find_rings_and_gaps: valid profile points =", length(flux_v), "/", length(profile.mean_flux))

    if ring_prominence === nothing
        ring_prominence = 3 * median(filter(!isnan, profile.std_flux))
    end
    gap_prominence === nothing && (gap_prominence = ring_prominence)
    _dbg("find_rings_and_gaps: ring_prominence =", ring_prominence, "gap_prominence =", gap_prominence)

    ring_idx, ring_proms = find_peaks(flux_v; prominence = ring_prominence)
    gap_idx, gap_proms = find_peaks(-flux_v; prominence = gap_prominence)
    _dbg("find_rings_and_gaps: raw ring_idx =", ring_idx, "raw gap_idx =", gap_idx)

    rings = [(radius_au = radius_v[i], flux = flux_v[i], prominence = p) for (i, p) in zip(ring_idx, ring_proms)]
    gaps  = [(radius_au = radius_v[i], flux = flux_v[i], prominence = p) for (i, p) in zip(gap_idx, gap_proms)]
    return rings, gaps
end

"""
    find_rings_and_gaps_residual(profile; window_frac=0.22, polyorder=3, ring_prominence=nothing,
                                 gap_prominence=nothing, prominence_sigma=1.2,
                                 min_radius_beams=2.5, min_abs_residual_frac=0.3)
        -> (rings, gaps, residual_data)

Locate ring/gap substructure using a smooth-baseline-subtraction approach
(as in e.g. Huang et al. 2018's DSHARP substructure analysis), rather than
`find_peaks` on the raw profile.

Why this is needed: `find_peaks` only flags genuine local maxima/minima.
When a disk's radial profile is dominated by a steep, monotonically
declining envelope (a bright unresolved central source falling off with
radius, as here), real ring/gap substructure often shows up only as
"shoulders" -- slope changes riding on top of the decline -- which never
turn over into a local max/min and so are invisible to plain `find_peaks`.
This function fits a smooth Savitzky-Golay baseline to the profile
(capturing just the broad monotonic falloff, not the substructure),
subtracts it to get a residual, and runs `find_peaks` on the residual.

window_frac: Savitzky-Golay window as a fraction of the number of valid
    profile bins (forced odd). Larger = smoother baseline = more sensitive
    to broad shoulders; smaller = baseline tracks the data more closely =
    less sensitive.
polyorder: polynomial order of the Savitzky-Golay fit (3 is a reasonable
    default for a smoothly declining profile with a few features).
prominence_sigma: multiplier on the residual's standard deviation used to
    derive ring/gap prominence thresholds when those are `nothing`. Lower
    values (e.g. 0.6-0.8) recover weaker, few-percent features (like the
    shallow gaps at ~6 au and ~44 au reported for TW Hya by Tsukagoshi et
    al. 2016 and Andrews et al. 2016) at the cost of more noise-driven false
    positives -- always sanity-check low-sigma detections against the plot
    and the literature rather than trusting them blindly.
min_radius_beams: exclude detections within this many synthesized-beam FWHMs
    (BMAJ) of the center. CLEAN deconvolution of a very bright, compact,
    high-dynamic-range core commonly leaves negative-bowl / restoration
    artifacts within the first few beam-widths of the peak, which can
    masquerade as a "gap". The baseline fit still uses the full profile;
    only the peak-finding *results* are filtered by radius. 0 disables.
min_abs_residual_frac: additionally require |residual| at the detected
    point to be at least this fraction of the residual's standard deviation.
    Prominence is relative (vs. neighbouring valleys/peaks), so a detection
    can be "prominent" while sitting right on the noise floor in absolute
    terms; this catches that case. 0 disables.
"""
function find_rings_and_gaps_residual(profile; window_frac = 0.22, polyorder = 3,
                                      ring_prominence = nothing, gap_prominence = nothing,
                                      prominence_sigma = 1.2, min_radius_beams = 2.5,
                                      min_abs_residual_frac = 0.3)
    valid = .!isnan.(profile.mean_flux)
    flux_v = profile.mean_flux[valid]
    radius_v = profile.radius_au[valid]

    n = length(flux_v)
    window = max(round(Int, window_frac * n), polyorder + 2)
    iseven(window) && (window += 1)
    window = min(window, isodd(n) ? n : n - 1)
    _dbg("find_rings_and_gaps_residual: n_valid =", n, "savgol window =", window, "polyorder =", polyorder)

    baseline = savgol_filter(flux_v, window, polyorder)
    residual = flux_v .- baseline
    bunit = profile.bunit
    residual_std = std(residual; corrected = false)          # population std, like np.std
    _dbg("find_rings_and_gaps_residual: residual std =", residual_std,
         "residual min/max =", minimum(residual), maximum(residual))

    ring_prominence === nothing && (ring_prominence = prominence_sigma * residual_std)
    gap_prominence === nothing && (gap_prominence = prominence_sigma * residual_std)
    _dbg("find_rings_and_gaps_residual: ring_prominence =", ring_prominence, "gap_prominence =", gap_prominence)

    ring_idx, ring_proms = find_peaks(residual; prominence = ring_prominence)
    gap_idx, gap_proms = find_peaks(-residual; prominence = gap_prominence)
    _dbg("find_rings_and_gaps_residual: raw ring_idx =", ring_idx, "raw gap_idx =", gap_idx)

    bmaj_au = profile.bmaj_au
    min_radius_au = (bmaj_au !== nothing && min_radius_beams > 0) ? min_radius_beams * bmaj_au : 0.0
    _dbg("find_rings_and_gaps_residual: bmaj_au =", bmaj_au, "min_radius_beams =", min_radius_beams,
         "-> min_radius_au =", min_radius_au)

    min_abs_residual = min_abs_residual_frac * residual_std
    _dbg("find_rings_and_gaps_residual: min_abs_residual_frac =", min_abs_residual_frac,
         "-> min_abs_residual =", min_abs_residual)

    keep(i) = radius_v[i] >= min_radius_au && abs(residual[i]) >= min_abs_residual

    all_idx = vcat(ring_idx, gap_idx)
    dropped_radius = [i for i in all_idx if radius_v[i] < min_radius_au]
    dropped_noise = [i for i in all_idx if radius_v[i] >= min_radius_au && abs(residual[i]) < min_abs_residual]
    if !isempty(dropped_radius)
        @printf("Excluded %d detection(s) within %.2f au (%sx beam FWHM) of center as likely beam/CLEAN artifacts:\n",
                length(dropped_radius), min_radius_au, min_radius_beams)
        for i in dropped_radius
            @printf("    r = %.1f au, residual = %.4e %s (%+.2f sigma)\n",
                    radius_v[i], residual[i], bunit, residual[i] / residual_std)
        end
    end
    if !isempty(dropped_noise)
        @printf("Excluded %d detection(s) with |residual| below %.2e %s (%sx residual std) as likely noise/boundary artifacts, despite passing the prominence test:\n",
                length(dropped_noise), min_abs_residual, bunit, min_abs_residual_frac)
        for i in dropped_noise
            @printf("    r = %.1f au, residual = %.4e %s (%+.2f sigma)\n",
                    radius_v[i], residual[i], bunit, residual[i] / residual_std)
        end
    end

    rings = [(radius_au = radius_v[i], flux = flux_v[i], residual = residual[i], prominence = p)
             for (i, p) in zip(ring_idx, ring_proms) if keep(i)]
    gaps = [(radius_au = radius_v[i], flux = flux_v[i], residual = residual[i], prominence = p)
            for (i, p) in zip(gap_idx, gap_proms) if keep(i)]
    residual_data = (radius_au = radius_v, flux = flux_v, baseline = baseline, residual = residual,
                     min_radius_au = min_radius_au, min_abs_residual = min_abs_residual,
                     distance_pc = profile.distance_pc, bunit = bunit)
    return rings, gaps, residual_data
end

"""
    report_residual_near(residual_data, target_radius_au; window_au=3.0)

Print the raw residual value(s) near a specific radius, independent of
whether `find_peaks` flagged anything there. Useful for checking a feature
that a prominence/noise-floor filter may have dropped (or that never
registered as a local extremum): "is there a real few-sigma dip near the
literature's 44 au gap even though nothing survived filtering there?" No
threshold is applied; it just reports what the residual actually does.
"""
function report_residual_near(residual_data, target_radius_au; window_au = 3.0)
    r_au = residual_data.radius_au
    res = residual_data.residual
    res_std = std(res; corrected = false)
    bunit = residual_data.bunit

    mask = abs.(r_au .- target_radius_au) .<= window_au
    if !any(mask)
        println("No profile bins found within $window_au au of $target_radius_au au.")
        return nothing
    end

    @printf("\nRaw residual near r = %s au (+/-%s au, residual std = %.4e %s):\n",
            target_radius_au, window_au, res_std, bunit)
    rm_, resm = r_au[mask], res[mask]
    dmin = minimum(abs.(rm_ .- target_radius_au))
    for (r, v) in zip(rm_, resm)
        marker = abs(r - target_radius_au) == dmin ? " <-- closest" : ""
        @printf("    r = %.2f au, residual = %.4e %s (%+.2f sigma)%s\n", r, v, bunit, v / res_std, marker)
    end

    ci = argmin(abs.(r_au .- target_radius_au))
    @printf("  Closest bin: r = %.2f au, residual = %.4e %s (%+.2f sigma)\n",
            r_au[ci], res[ci], bunit, res[ci] / res_std)
    return nothing
end

## Literature comparison and plots

In [ ]:
# ---------------------------------------------------------------------------
# Literature overlay + plots
# ---------------------------------------------------------------------------

const _LIT_LABEL = Dict(:hole => "central hole (lit.)", :deep => "deep gap (lit.)",
                        :shallow => "shallow gap (lit.)", :gap => "gap (lit.)")
const _LIT_STYLE = Dict(:hole => (color = ("#AAAAAA", 0.7), linewidth = 1.2),
                        :deep => (color = (C_LIT, 0.85), linewidth = 1.6),
                        :shallow => (color = (C_LIT, 0.55), linewidth = 1.1),
                        :gap => (color = (C_LIT, 0.75), linewidth = 1.3))

"""
    literature_features_au(t::AlmaTarget, distance_pc) -> Vector{(radius_au, label, kind)}

The target's literature gap radii rescaled from the distance the literature
assumed (`t.lit_distance_pc`) to the distance used in this run. Empty if the
catalogue has none for this target.
"""
function literature_features_au(t::AlmaTarget, distance_pc)
    isempty(t.lit) && return Tuple{Float64,String,Symbol}[]
    scale = distance_pc / (t.lit_distance_pc === nothing ? distance_pc : t.lit_distance_pc)
    return Tuple{Float64,String,Symbol}[(r * scale, _LIT_LABEL[kind], kind) for (r, kind) in t.lit]
end

"""
    compare_with_literature(lit, gaps, residual_data; bmaj_au=nothing)

Table of each literature feature against the nearest detected gap, in au and
in beams, plus the raw residual at the nearest bin when nothing was detected
within one beam.
"""
function compare_with_literature(lit, gaps, residual_data; bmaj_au = nothing)
    isempty(lit) && return nothing
    r_au, res = residual_data.radius_au, residual_data.residual
    res_std = std(res; corrected = false)
    tol = bmaj_au === nothing ? 2.0 : bmaj_au
    println("\nLiterature comparison (radii rescaled to the adopted distance; tolerance = 1 beam = $(round(tol; digits = 2)) au):")
    @printf("  %-20s %9s   %-22s %s\n", "feature", "lit [au]", "nearest detected gap", "delta [au] (beams)")
    for (r, label, kind) in sort(lit; by = first)
        if r < residual_data.min_radius_au
            @printf("  %-20s %9.1f   %-22s %s\n", label, r, "-", "inside the excluded beam/CLEAN zone")
            continue
        end
        if !isempty(gaps)
            gi = argmin([abs(g.radius_au - r) for g in gaps])
            d = gaps[gi].radius_au - r
            if abs(d) <= tol
                @printf("  %-20s %9.1f   %-22.1f %+.1f (%+.2f)\n", label, r, gaps[gi].radius_au, d, d / tol)
                continue
            end
        end
        ci = argmin(abs.(r_au .- r))
        @printf("  %-20s %9.1f   %-22s raw residual %.2e at r = %.1f au (%+.2f sigma)\n",
                label, r, "none within 1 beam", res[ci], r_au[ci], res[ci] / res_std)
    end
    return nothing
end

"""
    plot_radial_profile(profile; rings=[], gaps=[], title="", filename=...) -> Figure

Azimuthally averaged flux vs. radius, with rings/gaps marked.
"""
function plot_radial_profile(profile; rings = [], gaps = [], title = "Radial profile",
                             filename = "radial_profile.png")
    _dbg("plot_radial_profile: called with", length(rings), "rings and", length(gaps), "gaps")
    r, f, s = profile.radius_au, profile.mean_flux, profile.std_flux

    fig = Figure(size = (820, 520))
    ax = Axis(fig[1, 1]; xlabel = "Deprojected radius [au]", ylabel = "Flux density [$(profile.bunit)]",
              title = title, titlesize = 13)
    band!(ax, r, f .- s, f .+ s; color = (C_FLUX, 0.2), label = "±1σ scatter")
    lines!(ax, r, f; color = C_FLUX, linewidth = 2, label = "Azimuthal mean")

    if !isempty(rings)
        rr = [x.radius_au for x in rings]
        vlines!(ax, rr; color = C_RING, linestyle = :dash, linewidth = 1)
        scatter!(ax, rr, [x.flux for x in rings]; color = C_RING, marker = :utriangle, markersize = 13, label = "Ring")
    end
    if !isempty(gaps)
        gr = [x.radius_au for x in gaps]
        vlines!(ax, gr; color = C_GAP, linestyle = :dash, linewidth = 1)
        scatter!(ax, gr, [x.flux for x in gaps]; color = C_GAP, marker = :dtriangle, markersize = 13, label = "Gap")
    end
    axislegend(ax; position = :rt)

    display(fig)
    _save_fig(fig, filename)
    return fig
end

"""
    plot_residual_profile(residual_data; rings=[], gaps=[], lit_features=[], title="", filename=...) -> Figure

Two panels: raw profile with its Savitzky-Golay baseline (top) and the
residual with detected rings (^) and gaps (v) (bottom). `lit_features` is the
output of `literature_features_au` (dotted vertical lines).
"""
function plot_residual_profile(residual_data; rings = [], gaps = [], lit_features = [],
                               title = "Substructure (baseline-subtracted)",
                               filename = "substructure_residual.png")
    _dbg("plot_residual_profile: called with", length(rings), "rings and", length(gaps), "gaps")
    r = residual_data.radius_au
    min_radius_au = residual_data.min_radius_au

    fig = Figure(size = (820, 820))
    ax1 = Axis(fig[1, 1]; ylabel = "Flux density [$(residual_data.bunit)]", title = title, titlesize = 13)
    ax2 = Axis(fig[2, 1]; xlabel = "Deprojected radius [au]", ylabel = "Residual [$(residual_data.bunit)]")
    linkxaxes!(ax1, ax2)
    hidexdecorations!(ax1; grid = false)
    rowsize!(fig.layout, 1, Auto(1.2))
    rowsize!(fig.layout, 2, Auto(1.0))
    rowgap!(fig.layout, 8)

    lines!(ax1, r, residual_data.flux; color = C_FLUX, linewidth = 2, label = "Azimuthal mean")
    lines!(ax1, r, residual_data.baseline; color = "#CCCCCC", linewidth = 1.4, linestyle = :dash, label = "Smooth baseline")
    if min_radius_au > 0
        vspan!(ax1, [0.0], [min_radius_au]; color = ("#888888", 0.22),
               label = @sprintf("Excluded (<%.1f au, beam/CLEAN artifacts)", min_radius_au))
        vspan!(ax2, [0.0], [min_radius_au]; color = ("#888888", 0.22))
    end

    hlines!(ax2, 0.0; color = "#888888", linewidth = 0.8)
    lines!(ax2, r, residual_data.residual; color = C_RESID, linewidth = 2, label = "Residual")

    for kind in (:hole, :deep, :shallow, :gap)
        sel = [x for x in lit_features if x[3] == kind]
        isempty(sel) && continue
        for ax in (ax1, ax2)
            vlines!(ax, [x[1] for x in sel]; linestyle = :dot, label = sel[1][2], _LIT_STYLE[kind]...)
        end
    end

    if !isempty(rings)
        rr = [x.radius_au for x in rings]
        vlines!(ax2, rr; color = C_RING, linestyle = :dash, linewidth = 1)
        scatter!(ax2, rr, [x.residual for x in rings]; color = C_RING, marker = :utriangle, markersize = 15, label = "Ring")
    end
    if !isempty(gaps)
        gr = [x.radius_au for x in gaps]
        vlines!(ax2, gr; color = C_GAP, linestyle = :dash, linewidth = 1)
        scatter!(ax2, gr, [x.residual for x in gaps]; color = C_GAP, marker = :dtriangle, markersize = 15, label = "Gap")
    end

    axislegend(ax1; position = :rt, labelsize = 11)
    axislegend(ax2; position = :rt, labelsize = 11)

    display(fig)
    _save_fig(fig, filename)
    return fig
end

## Pipeline

In [ ]:
# ---------------------------------------------------------------------------
# Full pipeline for one target
# ---------------------------------------------------------------------------

"""
    run_alma_pipeline(target::AlmaTarget; kwargs...) -> NamedTuple

Query -> choose observation -> download product -> display -> radial profile ->
ring/gap detection -> literature comparison, for any `AlmaTarget`.

Selection / download
  proposal_id    restrict to one project (default: the catalogue's, usually none)
  band           e.g. 7 to keep only Band 7 observations
  obs_choice     row of the observation table to use (1 = finest resolution)
  public_only    only public data (default true)
  name_filter    match target_name against the target's aliases (default true)
  radius_deg     cone radius for the position query
  max_tar_gb     refuse tarballs larger than this unless force_download=true
Analysis (defaults come from the catalogue entry)
  distance_pc, incl_deg, pa_deg, max_radius_arcsec, n_bins
  center / center_mode   see `extract_radial_profile`
  prominence_sigma       ring/gap prominence threshold, in residual std (0.7)
  cutout_factor          image cutout = cutout_factor x max_radius (default 2.4)
"""
function run_alma_pipeline(target::AlmaTarget;
        proposal_id = target.proposal_id, band = nothing, obs_choice::Integer = 1,
        public_only::Bool = true, name_filter::Bool = true, radius_deg::Real = 0.05,
        max_tar_gb::Real = MAX_TAR_GB[], force_download::Bool = false,
        distance_pc = target.distance_pc, incl_deg = target.incl_deg, pa_deg = target.pa_deg,
        max_radius_arcsec = target.rmax_arcsec, n_bins = nothing,
        center = nothing, center_mode::Symbol = :optimize,
        prominence_sigma::Real = 0.7, cutout_factor::Real = 2.4,
        asinh_a::Real = 0.01, vmax_percentile::Real = 99.5)

    target = _ensure_coords(target)
    println("="^80)
    @printf("Target: %s   (RA %.4f, Dec %+.4f; %s)\n", target.name, target.ra_deg, target.dec_deg, target.note)
    println("="^80)
    target.distance_known || println("WARNING: no distance given; radii in au assume 100 pc (arcsec values are unaffected).")
    if incl_deg === nothing || pa_deg === nothing
        println("NOTE: inclination/PA unknown -> treated as face-on (no deprojection). Pass incl_deg= and pa_deg= to override.")
    end
    incl_deg = something(incl_deg, 0.0)
    pa_deg = something(pa_deg, 0.0)

    results = query_target(target; public_only = public_only, proposal_id = proposal_id,
                           name_filter = name_filter, radius_deg = radius_deg)
    nrow(results) == 0 && return (target = target, results = results)

    obs = summarize_observations(results)
    chosen = select_observation(obs; band = band, choice = obs_choice)
    chosen === nothing && return (target = target, results = results, observations = obs)
    member_ous_uid = String(chosen.member_ous_uid)

    data_info = inspect_products(member_ous_uid)
    tarball_url = pick_product_tarball(data_info)
    tarball_url === nothing && return (target = target, results = results, observations = obs, data_info = data_info)

    size_gb = tarball_size_gb(data_info, tarball_url)
    if size_gb !== nothing && size_gb > max_tar_gb && !force_download
        @printf("\nSTOP: %s is %.1f GB, above max_tar_gb = %.1f.\n", basename(tarball_url), size_gb, max_tar_gb)
        println("Re-run with force_download=true, raise max_tar_gb, or pick another observation with obs_choice=N / band=N.")
        return (target = target, results = results, observations = obs, data_info = data_info, tarball_url = tarball_url)
    end

    extracted = download_and_extract_fits(tarball_url)
    if isempty(extracted)
        println("No FITS images in this package (it may hold only measurement sets). " *
                "Try another observation with obs_choice=N, or the external_ari_l package.")
        return (target = target, results = results, observations = obs, data_info = data_info, tarball_url = tarball_url)
    end
    candidates = rank_fits_files(extracted)
    println("\nExtracted $(length(candidates)) FITS file(s), best candidate first:")
    foreach(f -> println("  ", f), candidates)
    local_path = candidates[1]
    _fits_score(local_path) >= 10 &&
        println("WARNING: the best candidate looks like a PSF/model/cube-type product, not a continuum image.")
    println("\nUsing: ", local_path)

    img = load_fits_image(local_path)
    label = target.name
    profile = rings = gaps = rings_r = gaps_r = residual_data = nothing
    lit_features = literature_features_au(target, distance_pc)
    try
        profile = extract_radial_profile(img; incl_deg = incl_deg, pa_deg = pa_deg, distance_pc = distance_pc,
                                         max_radius_arcsec = max_radius_arcsec, n_bins = n_bins,
                                         center = center, center_mode = center_mode)

        display_fits_image(img; label = label, center = profile.center_pix,
                           cutout_arcsec = cutout_factor * max_radius_arcsec,
                           asinh_a = asinh_a, vmax_percentile = vmax_percentile,
                           filename = "$(target.slug)_image.png")

        rings, gaps = find_rings_and_gaps(profile)
        println("\n[raw profile] Detected $(length(rings)) ring(s), $(length(gaps)) gap(s) with plain find_peaks.")
        plot_radial_profile(profile; rings = rings, gaps = gaps,
                            title = "$(label) radial profile — $(basename(local_path))",
                            filename = "$(target.slug)_radial_profile.png")

        # Rings/gaps usually appear only as shoulders on a steep decline, so
        # fit and subtract a smooth baseline and search the residual instead.
        rings_r, gaps_r, residual_data = find_rings_and_gaps_residual(profile; prominence_sigma = prominence_sigma)
        println("\n[baseline-subtracted] Detected $(length(rings_r)) ring(s):")
        for r in rings_r
            @printf("  r = %.1f au, residual = %.4e %s, prominence = %.2e\n", r.radius_au, r.residual, profile.bunit, r.prominence)
        end
        println("\n[baseline-subtracted] Detected $(length(gaps_r)) gap(s):")
        for g in gaps_r
            @printf("  r = %.1f au, residual = %.4e %s, prominence = %.2e\n", g.radius_au, g.residual, profile.bunit, g.prominence)
        end

        compare_with_literature(lit_features, gaps_r, residual_data; bmaj_au = profile.bmaj_au)

        plot_residual_profile(residual_data; rings = rings_r, gaps = gaps_r, lit_features = lit_features,
                              title = "$(label) continuum substructure — $(basename(local_path))",
                              filename = "$(target.slug)_substructure.png")
    catch err
        println("ERROR: analysis raised an exception:")
        showerror(stdout, err, catch_backtrace())
        println()
    end

    return (target = target, results = results, observations = obs, chosen = chosen, data_info = data_info,
            tarball_url = tarball_url, fits_files = candidates, local_path = local_path,
            profile = profile, rings = rings, gaps = gaps,
            rings_residual = rings_r, gaps_residual = gaps_r, residual_data = residual_data,
            lit_features = lit_features)
end

## Select and run
The first run for a target downloads its product tarball (several GB possible; only FITS files are kept). Packages larger than `MAX_TAR_GB` are refused unless `force_download = true`. Plots are saved to `./alma_plots/`.

In [ ]:
# Choose the target:
#   nothing            -> print the menu and prompt for a number or name
#   <number> or  "MWC 480"   -> pick directly (menu numbers are alphabetical; names are easier) (any spelling works: "mwc480")
#   custom_target("V883 Ori"; distance_pc = 388)   -> any other source (name resolved via Sesame)
TARGET_CHOICE = nothing

target = select_target(TARGET_CHOICE)
out = run_alma_pipeline(target);
# Useful options:  band = 7,  obs_choice = 2,  proposal_id = "2016.1.00484.L",
#                  center_mode = :peak,  incl_deg = 40, pa_deg = 90,  force_download = true

## Optional: re-tune the substructure detection

In [ ]:
# Re-run just the substructure detection with different settings -- no
# re-download needed. Note `min_abs_residual_frac` (default 0.3) can drop
# candidates that pass the prominence test; try 0.1 to see weak features.
if out !== nothing && hasproperty(out, :profile) && out.profile !== nothing
    rings2, gaps2, rd2 = find_rings_and_gaps_residual(out.profile; prominence_sigma = 0.6,
                                                       min_abs_residual_frac = 0.1)
    compare_with_literature(out.lit_features, gaps2, rd2; bmaj_au = out.profile.bmaj_au)
    plot_residual_profile(rd2; rings = rings2, gaps = gaps2, lit_features = out.lit_features,
        title = "$(out.target.name) substructure (retuned)",
        filename = "$(out.target.slug)_substructure_retuned.png");
end

## Optional: rank targets by archive observation count

In [ ]:
# OPTIONAL: the 100 most-observed science targets in the ALMA archive, listed alphabetically.
# Name variants are merged, OFF positions/Sun dropped, and curated targets (marked *) keep their
# distance / inclination / PA. Uncurated ones assume 100 pc unless you pass distance_pc.
# top_archive_targets(100; sort_by = :rows) lists by archive rows instead.
archive_top = top_archive_targets(100)
# out2 = run_alma_pipeline(archive_top[3]; distance_pc = 150.0)
